<a href="https://colab.research.google.com/github/tokakhaled/AISA-ArabicFC/blob/main/baseline_pp_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install -U transformers datasets accelerate
import torch, re, json
from transformers import AutoTokenizer, AutoModelForCausalLM

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 22.3 MB/s eta 0:00:00


In [ ]:
# ===== AISA-ArabicFC: full baseline inference (corrected) =====
import torch, re, json
from collections import Counter
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from tqdm import tqdm

assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4, then Restart session."

In [ ]:
MODEL_ID = "TuwaiqAcademy/AISA-AR-FunctionCall-Think"

# --- load model in bfloat16 (NOT fp16 - Gemma breaks in fp16) ---
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16
).to("cuda").eval()
print("device:", next(model.parameters()).device, "| dtype:", next(model.parameters()).dtype)

# --- parser: raw generation -> {function_name, arguments, think} ---
def parse_model_output(text):
    out = {"function_name": "none", "arguments": {}, "think": ""}
    if (m := re.search(r"<think>\s*(.*?)\s*</think>", text, re.DOTALL)):
        out["think"] = m.group(1).strip()
    if (m := re.search(r"<start_function_call>\s*call:(\w+)\{(.*?)\}\s*<end_function_call>", text, re.DOTALL)):
        out["function_name"] = m.group(1)
        for key, sval, nval in re.findall(r"(\w+):(?:<escape>(.*?)<escape>|([^,}]+))", m.group(2)):
            val = sval if sval else nval
            try: val = float(val) if "." in str(val) else int(val)
            except (ValueError, TypeError): pass
            out["arguments"][key] = val
    return out

# --- build prompts (robust split on the model turn) ---
SEP = "<start_of_turn>model"
val = load_dataset("TuwaiqAcademy/AISA-ArabicFC", split="dev")
def make_prompt(t):
    head = t.split(SEP)[0]
    return head + SEP + "\n"
prompts = [make_prompt(r["text"]) for r in val]
# sanity: gold answer must NOT be inside the prompt
assert "<start_function_call>" not in prompts[2], "PROMPT SPLIT BROKEN: gold is leaking into the input!"

# --- batched generation ---
BATCH = 16
preds = []
for s in tqdm(range(0, len(prompts), BATCH)):
    chunk = prompts[s:s+BATCH]
    inp = tok(chunk, return_tensors="pt", padding=True, truncation=True,
              max_length=4096, add_special_tokens=False).to("cuda")
    with torch.no_grad():
        gen = model.generate(**inp, max_new_tokens=250, do_sample=False,
                             pad_token_id=tok.pad_token_id)
    for j in range(len(chunk)):
        raw = tok.decode(gen[j][inp["input_ids"].shape[1]:], skip_special_tokens=False)
        p = parse_model_output(raw)
        preds.append({"id": s+j, "tool_called": p["function_name"],
                      "arguments": p["arguments"], "think": p["think"]})

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/63.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/714 [00:00<?, ?B/s]

...

data/train-00000-of-00001.parquet:   0%|          | 0.00/10.6M [00:00<?, ?B/s]

data/dev-00000-of-00001.parquet:   0%|          | 0.00/678k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10550 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/545 [00:00<?, ? examples/s]

100%|██████████| 35/35 [06:05<00:00, 10.44s/it]


In [ ]:

# --- write submissions ---
with open("submission_trackA.jsonl", "w") as f:
    for p in preds:
        f.write(json.dumps({"id": p["id"], "tool_called": p["tool_called"],
                            "arguments": p["arguments"]}, ensure_ascii=False) + "\n")
with open("submission_trackB.jsonl", "w") as f:
    for p in preds:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

# --- health check ---
c = Counter(p["tool_called"] for p in preds)
print("\nwrote", len(preds), "predictions")
print("none:", c["none"], "/", len(preds), "  (should be SMALL, ~a few %)")
print("top calls:", c.most_common(6))
print("\nfirst 3 NON-none examples:")
shown = 0
for p in preds:
    if p["tool_called"] != "none":
        print(json.dumps({k: p[k] for k in ("id","tool_called","arguments")}, ensure_ascii=False))
        shown += 1
        if shown == 3: break



wrote 545 predictions
none: 58 / 545   (should be SMALL, ~a few %)
top calls: [('none', 58), ('book_doctor_appointment', 30), ('get_weather', 29), ('search_medications', 27), ('get_qibla_direction', 25), ('translate_text', 25)]

first 3 NON-none examples:
{"id": 0, "tool_called": "check_traffic_violations", "arguments": {"id_number": 987654321}}
{"id": 1, "tool_called": "get_qibla_direction", "arguments": {"city": "الجيزة"}}
{"id": 2, "tool_called": "convert_currency", "arguments": {"amount": 1500.0, "from_currency": "SAR", "to_currency": "USD"}}


In [ ]:
# ===== local scoring: FnAcc + approx ArgEM on dev =====
import unicodedata
AR2EN = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")
def norm(v):
    s = str(v).translate(AR2EN).strip().lower()
    s = "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))
    s = s.replace("أ","ا").replace("إ","ا").replace("آ","ا").replace("ى","ي").replace("ة","ه")
    try:
        f = float(s); s = str(int(f)) if f == int(f) else str(f)
    except ValueError: pass
    return s

def norm_args(d):
    return {norm(k): norm(v) for k, v in d.items()}

# gold from the dev text field
gold = []
for r in val:
    g = parse_model_output(r["text"].split(SEP, 1)[1])
    gold.append({"fn": g["function_name"], "args": g["arguments"]})

fn_correct = 0
arg_correct = 0
pos = 0
for p, g in zip(preds, gold):
    if p["tool_called"] == g["fn"]:
        fn_correct += 1
    if g["fn"] != "none":              # ArgEM scored on positive gold only
        pos += 1
        ga = norm_args(g["args"])
        pa = norm_args(p["arguments"])
        # every gold key must match (ignore extra predicted keys)
        if all(k in pa and pa[k] == v for k, v in ga.items()) and len(pa) >= len(ga):
            arg_correct += 1

FnAcc = fn_correct / len(preds)
ArgEM = arg_correct / pos
OverallA = 0.40*FnAcc + 0.60*ArgEM
print(f"FnAcc    : {FnAcc:.3f}")
print(f"ArgEM    : {ArgEM:.3f}   (approx)")
print(f"Overall A: {OverallA:.3f}   = 0.40*FnAcc + 0.60*ArgEM")
print(f"(positives scored: {pos})")


FnAcc    : 0.976
ArgEM    : 0.596   (approx)
Overall A: 0.748   = 0.40*FnAcc + 0.60*ArgEM
(positives scored: 500)


In [ ]:
# ===== local scoring: Track B =====
import unicodedata

def is_arabic(s):
    return any("\u0600" <= ch <= "\u06FF" for ch in s)

fn_correct = arg_correct = pos = think_ok = 0
for p, g in zip(preds, gold):
    # FnAcc (all samples)
    if p["tool_called"] == g["fn"]:
        fn_correct += 1
    # ArgEM (positive gold only)
    if g["fn"] != "none":
        pos += 1
        ga, pa = norm_args(g["args"]), norm_args(p["arguments"])
        if all(k in pa and pa[k] == v for k, v in ga.items()) and len(pa) >= len(ga):
            arg_correct += 1
    # ThinkRate: non-empty Arabic reasoning trace
    t = p.get("think", "")
    if t and is_arabic(t):
        think_ok += 1

FnAcc     = fn_correct / len(preds)
ArgEM     = arg_correct / pos
ThinkRate = think_ok / len(preds)
OverallB  = 0.30*FnAcc + 0.50*ArgEM + 0.20*ThinkRate

print(f"FnAcc    : {FnAcc:.3f}")
print(f"ArgEM    : {ArgEM:.3f}   (approx)")
print(f"ThinkRate: {ThinkRate:.3f}")
print(f"Overall B: {OverallB:.3f}   = 0.30*FnAcc + 0.50*ArgEM + 0.20*ThinkRate")
print(f"(positives: {pos},  samples with think: {think_ok}/{len(preds)})")

FnAcc    : 0.976
ArgEM    : 0.596   (approx)
ThinkRate: 0.890
Overall B: 0.769   = 0.30*FnAcc + 0.50*ArgEM + 0.20*ThinkRate
(positives: 500,  samples with think: 485/545)


In [ ]:
# ===== post-processing: deterministic fixes, then re-score =====
import re

# Arabic/English language name -> ISO 639-1
LANG = {}
for code, names in {
 "de":["german","الالمانيه","الماني","الالماني"], "en":["english","الانجليزيه","انجليزي"],
 "fr":["french","الفرنسيه","فرنسي"], "es":["spanish","الاسبانيه","اسباني"],
 "ar":["arabic","العربيه","عربي"], "tr":["turkish","التركيه","تركي"],
 "zh":["chinese","الصينيه","صيني"], "ru":["russian","الروسيه","روسي"],
 "ur":["urdu","الارديه","اردو"], "fa":["persian","farsi","الفارسيه","فارسي"],
 "hi":["hindi","الهنديه","هندي"], "it":["italian","الايطاليه","ايطالي"],
}.items():
    for n in names: LANG[norm(n)] = code

ZTYPE = {norm("ذهب"):"gold", norm("فضه"):"silver", norm("فضة"):"silver",
         norm("مال"):"cash", norm("نقد"):"cash", norm("نقود"):"cash"}

def postprocess(fn, args, query):
    a = dict(args)
    if "target_language" in a:
        a["target_language"] = LANG.get(norm(a["target_language"]), a["target_language"])
    if fn == "calculate_zakat" and "type" in a:
        a["type"] = ZTYPE.get(norm(a["type"]), a["type"])
    if "recipient_iban" in a:                      # drop invented IBANs
        iban = str(a["recipient_iban"])
        if iban not in query and not re.search(r"[A-Z]{2}\d{2}[A-Z0-9]{8,}", query) and not re.search(r"\d{15,}", query):
            a.pop("recipient_iban")
    if "currency" in a and re.fullmatch(r"[a-z]{2,4}", str(a["currency"])):  # garbage like 'ghg'
        a.pop("currency")
    return a

def score(predictions):
    fn_c = arg_c = pos = 0
    for p, g in zip(predictions, gold):
        if p["tool_called"] == g["fn"]: fn_c += 1
        if g["fn"] != "none":
            pos += 1
            ga, pa = norm_args(g["args"]), norm_args(p["arguments"])
            if all(k in pa and pa[k]==v for k,v in ga.items()) and len(pa) >= len(ga):
                arg_c += 1
    return fn_c/len(predictions), arg_c/pos

# apply
preds_pp = []
for p, r in zip(preds, val):
    q = r["text"]
    preds_pp.append({**p, "arguments": postprocess(p["tool_called"], p["arguments"], q)})

f0, a0 = score(preds)
f1, a1 = score(preds_pp)
print(f"BEFORE  FnAcc {f0:.3f}  ArgEM {a0:.3f}  OverallA {0.4*f0+0.6*a0:.3f}")
print(f"AFTER   FnAcc {f1:.3f}  ArgEM {a1:.3f}  OverallA {0.4*f1+0.6*a1:.3f}")
print(f"ArgEM delta: {a1-a0:+.3f}")

BEFORE  FnAcc 0.976  ArgEM 0.596  OverallA 0.748
AFTER   FnAcc 0.976  ArgEM 0.626  OverallA 0.766
ArgEM delta: +0.030


In [ ]:

# ===== Canonicalizer: Arabic enum values -> gold's canonical English/ISO forms =====
ZAKAT_TYPE = {
 "cash":["مال","المال","نقد","نقود","كاش","اموال","سيوله"],
 "gold":["ذهب","الذهب","ذهبي","سبائك ذهب"],
 "silver":["فضه","الفضه","فضي"],
 "trade":["تجاره","التجاره","عروض تجاريه","بضاعه","عروض التجاره","تجاري"],
 "fitr":["فطر","الفطر","فطره","زكاه الفطر"],
 "crops":["زروع","الزروع","محاصيل","زرع","حبوب","زراعه"],
 "general":["عام","عامه"],
 "salary":["راتب","الراتب","رواتب","دخل","المرتب"],
 "livestock":["مواشي","انعام","ماشيه","بهايم","ابل","اغنام"],
 "realestate":["عقار","عقارات","العقار","عقاري"],
 "shares":["اسهم","سهم","الاسهم"],
}
TERMINATION = {
 "resignation":["استقاله","استقال","استقلت","استقالت"],
 "dismissal":["فصل","طرد","طردت","فصلت","مفصول","تسريح","صرف"],
 "unfair_dismissal":["فصل تعسفي","طرد تعسفي","فصل جاير","تعسفي"],
 "end_of_contract":["انهاء عقد","انتهاء العقد","انتهاء عقد","نهايه العقد","انهاء العقد","انتهاء المده"],
 "retirement":["تقاعد","التقاعد","معاش","تقاعدت","احاله للتقاعد"],
 "economic":["اقتصادي","اسباب اقتصاديه","اقتصاديه"],
 "disciplinary":["تاديبي","تاديبيه","عقوبه تاديبيه","مخالفه"],
 "mutual_consent":["اتفاق","تراضي","اتفاق الطرفين","بالتراضي","اتفاق متبادل","توافق"],
}
LANG_SYN = {   # renamed to NOT clobber the LANG in the post-processing cell
 "en":["انجليزي","الانجليزيه","انكليزي","english"],
 "fr":["فرنسي","الفرنسيه","french"],
 "es":["اسباني","الاسبانيه","spanish"],
 "de":["الماني","الالمانيه","german"],
 "it":["ايطالي","الايطاليه","italian"],
 "ar":["عربي","العربيه","arabic"],
 "zh":["صيني","الصينيه","chinese","مندرين"],
 "ja":["ياباني","اليابانيه","japanese"],
 "tr":["تركي","التركيه","turkish"],
 "fa":["فارسي","الفارسيه","persian","farsi"],
 "ko":["كوري","الكوريه","korean"],
}
def _build(m):
    d = {}
    for canon, syns in m.items():
        d[norm(canon)] = canon
        for s in syns: d[norm(s)] = canon
    return d
LK_ZAKAT, LK_TERM, LK_LANG = _build(ZAKAT_TYPE), _build(TERMINATION), _build(LANG_SYN)

def _canon(lk, val):
    n = norm(val)
    if n in lk: return lk[n]
    for key, canon in lk.items():
        if key and key in n: return canon      # substring fallback
    return val

def canonicalize(fn, args):
    a = dict(args)
    if fn == "calculate_zakat" and "type" in a:
        a["type"] = _canon(LK_ZAKAT, a["type"])
    if fn == "calculate_end_of_service" and "termination_type" in a:
        a["termination_type"] = _canon(LK_TERM, a["termination_type"])
    if fn == "translate_text" and "target_language" in a:
        a["target_language"] = _canon(LK_LANG, a["target_language"])
    return a

# apply canonicalize ON TOP of postprocess, then re-score  (this notebook: dev set = `val`, score() returns 2 values)
preds_full = []
for p, r in zip(preds, val):
    args = postprocess(p["tool_called"], p["arguments"], r["text"])
    args = canonicalize(p["tool_called"], args)
    preds_full.append({**p, "arguments": args})

f2, a2 = score(preds_full)
print(f"+PP        FnAcc 0.976  ArgEM 0.626  OverallA 0.766")
print(f"+PP+CANON  FnAcc {f2:.3f}  ArgEM {a2:.3f}  OverallA {0.4*f2 + 0.6*a2:.3f}")

+PP        FnAcc 0.976  ArgEM 0.626  OverallA 0.766
+PP+CANON  FnAcc 0.976  ArgEM 0.654  OverallA 0.783


In [ ]:
import json
from google.colab import files

# Track A (post-processed predictions = our best, 0.766)
with open("submission_trackA.jsonl", "w") as f:
    # for p in preds_pp:
    for p in preds_full:
        f.write(json.dumps({"id": p["id"], "tool_called": p["tool_called"],
                            "arguments": p["arguments"]}, ensure_ascii=False) + "\n")

# Track B (adds the think trace)
with open("submission_trackB.jsonl", "w") as f:
    # for p in preds_pp:
    for p in preds_full:
        f.write(json.dumps({"id": p["id"], "tool_called": p["tool_called"],
                            "arguments": p["arguments"], "think": p.get("think","")}, ensure_ascii=False) + "\n")

print("rows:", len(preds_pp))
files.download("submission_trackA.jsonl")


rows: 545


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download("submission_trackB.jsonl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



# Error Analysis

In [ ]:
# ===== error analysis: where does ArgEM fail? =====
import re as _re
from collections import Counter

def user_query(text):
    m = _re.search(r"<start_of_turn>user\s*(.*?)\s*<end_of_turn>", text, _re.DOTALL)
    return m.group(1).strip() if m else "?"

cats = Counter()
examples = {"wrong_function":[], "missing_key":[], "extra_key":[], "wrong_value":[]}

for p, g, r in zip(preds, gold, val):
    if g["fn"] == "none":
        continue
    ga, pa = norm_args(g["args"]), norm_args(p["arguments"])
    ok = all(k in pa and pa[k]==v for k,v in ga.items()) and len(pa) >= len(ga)
    if ok:
        continue
    if p["tool_called"] != g["fn"]:
        cat = "wrong_function"
    elif any(k not in pa for k in ga):
        cat = "missing_key"
    elif len(pa) > len(ga):
        cat = "extra_key"
    else:
        cat = "wrong_value"
    cats[cat] += 1
    if len(examples[cat]) < 4:
        examples[cat].append({
            "q": user_query(r["text"])[:160],
            "gold": {g["fn"]: g["args"]},
            "pred": {p["tool_called"]: p["arguments"]},
        })

total_wrong = sum(cats.values())
print(f"total positive failures: {total_wrong} / 500\n")
for k, n in cats.most_common():
    print(f"{k:16s}: {n}")

import json
for cat in ["wrong_value","missing_key","extra_key","wrong_function"]:
    print(f"\n===== {cat} examples =====")
    for e in examples[cat]:
        print("Q   :", e["q"])
        print("GOLD:", json.dumps(e["gold"], ensure_ascii=False))
        print("PRED:", json.dumps(e["pred"], ensure_ascii=False))
        print()

total positive failures: 202 / 500

wrong_value     : 156
extra_key       : 21
wrong_function  : 13
missing_key     : 12

===== wrong_value examples =====
Q   : كيف بحسب زكاة المال إذا عندي ٥٠٠٠٠ ليرة سورية؟
GOLD: {"calculate_zakat": {"amount": 50000, "currency": "SYP", "type": "cash"}}
PRED: {"calculate_zakat": {"amount": 50000.0, "currency": "سورية", "type": "مال"}}

Q   : ممكن تترجم ده للألماني؟ أنا مشغول جدا النهاردة
GOLD: {"translate_text": {"target_language": "de", "text": "أنا مشغول جدا النهاردة"}}
PRED: {"translate_text": {"target_language": "German", "text": "ممكن تترجم ده للألماني؟"}}

Q   : قارن لي أسعار هاتف iPhone 13 في مصر والسعودية
GOLD: {"compare_prices": {"country": "مصر", "product_name": "iPhone 13"}}
PRED: {"compare_prices": {"country": "مصر و السعودية", "product_name": "iPhone13"}}

Q   : أريد حجز موعد مع طبيب الأطفال في أبو ظبي يوم 15 أكتوبر
GOLD: {"book_doctor_appointment": {"city": "أبو ظبي", "date": "15 أكتوبر", "specialty": "طب الأطفال"}}
PRED: {"book_doctor_ap

In [ ]:
# ===== Analysis 1: function frequency + argument schema =====
from collections import Counter

# Function frequency (gold)
fn_counts = Counter(g["fn"] for g in gold)
print("=== Gold function frequency (top 30) ===")
for fn, n in fn_counts.most_common(30):
    print(f"{fn:30s} : {n}")

# Argument keys per function (gold)
schema_raw = {}
for g in gold:
    if g["fn"] == "none":
        continue
    fn = g["fn"]
    keys = set(g["args"].keys())
    schema_raw.setdefault(fn, set()).update(keys)

print("\n=== Gold argument keys per function ===")
for fn, keys in sorted(schema_raw.items(), key=lambda x: x[0]):
    print(f"{fn:30s} : {sorted(keys)}")

=== Gold function frequency (top 30) ===
none                           : 45
book_doctor_appointment        : 30
get_weather                    : 29
search_medications             : 27
check_insurance_coverage       : 27
translate_text                 : 26
search_hotels                  : 26
get_qibla_direction            : 25
check_iqama_status             : 25
check_traffic_violations       : 24
convert_currency               : 24
calculate_zakat                : 24
search_quran                   : 24
compare_prices                 : 24
calculate_customs              : 24
transfer_money                 : 24
order_food                     : 24
check_visa_status              : 24
search_umrah_packages          : 23
calculate_end_of_service       : 23
get_air_quality                : 23

=== Gold argument keys per function ===
book_doctor_appointment        : ['city', 'date', 'specialty']
calculate_customs              : ['category', 'currency', 'destination_country', 'product_value']
c

In [ ]:
# ===== Analysis 2: error categories per function =====
from collections import Counter

fn_error_cats = {}  # fn -> Counter({cat: count})

for p, g, r in zip(preds, gold, val):
    if g["fn"] == "none":
        continue

    ga = norm_args(g["args"])
    pa = norm_args(p["arguments"])

    ok = all(k in pa and pa[k] == v for k, v in ga.items()) and len(pa) >= len(ga)
    if ok:
        continue

    # same logic you used in error analysis cell
    if p["tool_called"] != g["fn"]:
        cat = "wrong_function"
    elif any(k not in pa for k in ga):
        cat = "missing_key"
    elif len(pa) > len(ga):
        cat = "extra_key"
    else:
        cat = "wrong_value"

    fn_error_cats.setdefault(g["fn"], Counter())[cat] += 1

print("=== Error categories per function (only functions with errors) ===")
for fn, ctr in sorted(fn_error_cats.items(), key=lambda x: fn_counts.get(x[0], 0), reverse=True):
    total = sum(ctr.values())
    cats_str = ", ".join(f"{k}: {v} ({v/total:.2%})" for k, v in ctr.items())
    print(f"{fn:30s} : total errors {total:3d} | {cats_str}")

=== Error categories per function (only functions with errors) ===
book_doctor_appointment        : total errors  22 | wrong_value: 20 (90.91%), extra_key: 2 (9.09%)
get_weather                    : total errors   7 | wrong_value: 4 (57.14%), missing_key: 3 (42.86%)
search_medications             : total errors   8 | wrong_value: 7 (87.50%), extra_key: 1 (12.50%)
check_insurance_coverage       : total errors  12 | wrong_value: 6 (50.00%), wrong_function: 2 (16.67%), extra_key: 4 (33.33%)
translate_text                 : total errors   7 | wrong_value: 6 (85.71%), wrong_function: 1 (14.29%)
search_hotels                  : total errors  14 | wrong_value: 11 (78.57%), wrong_function: 1 (7.14%), extra_key: 2 (14.29%)
check_iqama_status             : total errors   1 | wrong_function: 1 (100.00%)
get_qibla_direction            : total errors   1 | wrong_value: 1 (100.00%)
calculate_zakat                : total errors  24 | wrong_value: 22 (91.67%), extra_key: 2 (8.33%)
compare_prices      

In [ ]:
# ===== Analysis 3: sample user queries per high-frequency function =====
def show_examples_for(fn, max_examples=8):
    print(f"\n===== {fn} (examples) =====")
    shown = 0
    for g, r in zip(gold, val):
        if g["fn"] != fn:
            continue
        q = user_query(r["text"])
        print("-", q)
        shown += 1
        if shown == max_examples:
            break

# pick some functions to inspect (top by frequency)
top_fns = [fn for fn, _ in fn_counts.most_common(15) if fn != "none"]
for fn in top_fns:
    show_examples_for(fn)


===== book_doctor_appointment (examples) =====
- أريد حجز موعد مع طبيب الأطفال في أبو ظبي يوم 15 أكتوبر
- احجز لي موعدًا مع طبيب نسائية في الرياض الأسبوع القادم
- نريد نحجز موعد طبي مع طبيب قلب في العاصمة النهار السبت
- أحتاج أحجز موعد عند دكتور أسنان في جدة يوم الخميس
- ممكن أحجز مع دكتور قلب في المنصورة الأسبوع الجاي
- ممكن تحددلي معاد مع دكتور عيون في الجيزة الأسبوع الجاي؟
- أبي أحجز موعد مع دكتور عيون في الرياض بعد باجر
- أحتاج إلى استشارة مع طبيب أمراض جلدية في مكة يوم الأربعاء.

===== get_weather (examples) =====
- اريد معرفة حالة الطقس في بيروت لأسبوع كامل.
- ممكن تخبرني عن حالة الجو في مسقط اليوم؟
- ودي أعرف حالة الجو في المدينة المنورة لمدة أسبوع
- ما هي حالة الطقس في القاهرة لمدة ثلاثة أيام؟
- ما هي توقعات الطقس للأسبوع القادم في بيروت؟
- الجو في البحرين الاسبوع الجاي؟
- ما هو طقس دمشق اليوم؟
- أحتاج لمعرفة حالة الطقس في عمان لثلاثة أيام قادمة.

===== search_medications (examples) =====
- عندكوا بنادول في الصيدلية؟
- فين أقدر ألاقي برشام بانادول
- ابحث لي عن دواء البنادول
- 

In [ ]:
# ===== Analysis 4: inspect argument mismatches =====
mismatches = []  # (fn, key, gold_val, pred_val, query_snippet)

for p, g, r in zip(preds, gold, val):
    if g["fn"] == "none":
        continue
    ga = norm_args(g["args"])
    pa = norm_args(p["arguments"])
    for k in ga:
        if k in pa and ga[k] != pa[k]:
            q = user_query(r["text"])
            mismatches.append((g["fn"], k, ga[k], pa[k], q))

print(f"Total normalized mismatches: {len(mismatches)}")

# show a few examples
for fn, k, gv, pv, q in mismatches[:40]:
    print(f"\nfn={fn}, key={k}")
    print(f"  GOLD : {gv}")
    print(f"  PRED : {pv}")
    print(f"  Q    : {q[:160]}")

Total normalized mismatches: 223

fn=calculate_zakat, key=currency
  GOLD : syp
  PRED : سوريه
  Q    : كيف بحسب زكاة المال إذا عندي ٥٠٠٠٠ ليرة سورية؟

fn=calculate_zakat, key=type
  GOLD : cash
  PRED : مال
  Q    : كيف بحسب زكاة المال إذا عندي ٥٠٠٠٠ ليرة سورية؟

fn=translate_text, key=target_language
  GOLD : de
  PRED : german
  Q    : ممكن تترجم ده للألماني؟ أنا مشغول جدا النهاردة

fn=translate_text, key=text
  GOLD : انا مشغول جدا النهارده
  PRED : ممكن تترجم ده للالماني؟
  Q    : ممكن تترجم ده للألماني؟ أنا مشغول جدا النهاردة

fn=compare_prices, key=country
  GOLD : مصر
  PRED : مصر و السعوديه
  Q    : قارن لي أسعار هاتف iPhone 13 في مصر والسعودية

fn=compare_prices, key=product_name
  GOLD : iphone 13
  PRED : iphone13
  Q    : قارن لي أسعار هاتف iPhone 13 في مصر والسعودية

fn=book_doctor_appointment, key=date
  GOLD : 15 اكتوبر
  PRED : 2023-10-15
  Q    : أريد حجز موعد مع طبيب الأطفال في أبو ظبي يوم 15 أكتوبر

fn=book_doctor_appointment, key=specialty
  GOLD : طب الاطفال
  PRE

In [ ]:
# ===== Analysis 5: crude dialect heuristic =====
def guess_dialect(q):
    s = norm(q)
    # very rough and incomplete heuristics
    if any(w in s for w in ["عايز", "مش", "بتاع", "نهارده", "مصري"]):
        return "masri"
    if any(w in s for w in ["بدي", "لي", "إيش", "شو", "ما بدّي"]):
        return "levant"
    if any(w in s for w in ["بغي", "غا", "درهم", "طنجه"]):
        return "maghreb"
    if any(w in s for w in ["ابي", "وش", "الرياض", "جده", "رقم الاقامه"]):
        return "gulf"
    return "other"

dialect_counts = Counter()
for r in val:
    q = user_query(r["text"])
    dialect_counts[guess_dialect(q)] += 1

print("=== Approx dialect distribution (dev) ===")
for d, n in dialect_counts.most_common():
    print(f"{d:8s} : {n}")

=== Approx dialect distribution (dev) ===
levant   : 218
other    : 207
masri    : 70
gulf     : 32
maghreb  : 18
